# Eksperimen 02: Optimasi MASE & Feature Fusion (LightGBM + CatBoost Ensemble)

Notebook ini mengeksekusi rencana komprehensif untuk memenangkan kompetisi peramalan (*forecasting*) tiket bioskop **(D4–D10)** dengan mengoptimalkan metrik **Mean Absolute Scaled Error (MASE)**.

---

### 🌟 Ringkasan Pilar Arsitektur & Strategi:
1. **Optimasi Target Formulasi MASE**:
   $$\text{MASE} = \frac{1}{N} \sum_{i=1}^N \frac{|y_i - \hat{y}_i|}{S_{p_i}}$$
   Dengan mendefinisikan target terskala $y_{\text{scaled}} = \frac{y}{S_p}$, pelatihan model dengan fungsi kerugian **MAE (L1 Loss)** secara matematis meminimalkan metrik MASE secara langsung!
2. **Penyelesaian Bug Fatal Data Uji**:
   Fitur profil historis (D1–D3) pada data uji `test.csv` diekstrak dari **`test_history.csv`**, menjamin nilai fitur memori terisi penuh dan valid.
3. **Integrasi Multi-Sumber (*Feature Fusion*)**:
   - **Fitur Endogen**: Tiket D1, D2, D3, total tiket 3 hari pertama, skala $S_p$, rasio momentum ($D_2/D_1$, $D_3/D_2$, $D_3/D_1$), tren ($D_3 - D_1$), serta pergerakan okupansi dan kapasitas penayangan.
   - **Fitur Eksogen**: Kalender (hari ke-4 s.d 10, Jumat & Akhir pekan), hari libur (`holidays.csv`), batas harga tiket (`ticket_prices.csv`), potensi omset, dan metadata film (`movies.csv`).
   - **Dinamika Peluruhan**: Interaksi kurva peluruhan (*decay*) dan pengali lonjakan akhir pekan (*weekend multiplier*).
4. **Skema Validasi Bebas Kebocoran**: **GroupKFold (5-Fold)** berbasis `movie_title`.
5. **Hybrid Ensembling**: Kombinasi **LightGBM (L1 Regression)** dan **CatBoost (MAE)** untuk stabilitas prediksi optimal.

## 1. Import Library & Setup Environment

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import lightgbm as lgb
from catboost import CatBoostRegressor
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_absolute_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 100)

# Kunci Seed Reproducibility
SEED = 42
np.random.seed(SEED)
print(f"Environment siap. SEED terkunci pada: {SEED}")

ModuleNotFoundError: No module named 'catboost'

## 2. Load Data Multi-Sumber

In [ ]:
possible_paths = ['../../../dataset']
data_dir = None
for p in possible_paths:
    if os.path.exists(os.path.join(p, 'train.csv')):
        data_dir = p
        break

if data_dir is None:
    raise FileNotFoundError('Direktori dataset tidak ditemukan!')

print(f'>> Memuat data dari direktori: {data_dir}')
train = pd.read_csv(os.path.join(data_dir, 'train.csv'))
test = pd.read_csv(os.path.join(data_dir, 'test.csv'))
test_history = pd.read_csv(os.path.join(data_dir, 'test_history.csv'))
movies = pd.read_csv(os.path.join(data_dir, 'movies.csv'))
holidays = pd.read_csv(os.path.join(data_dir, 'holidays.csv'))
ticket_prices = pd.read_csv(os.path.join(data_dir, 'ticket_prices.csv'))

# Parsing Datetime
train['date_show'] = pd.to_datetime(train['date_show'])
test['date_show'] = pd.to_datetime(test['date_show'])
test_history['date_show'] = pd.to_datetime(test_history['date_show'])
holidays['date'] = pd.to_datetime(holidays['date'])

print(f'Train shape        : {train.shape}')
print(f'Test shape         : {test.shape}')
print(f'Test History shape : {test_history.shape}')
print(f'Movies shape       : {movies.shape}')
print(f'Holidays shape     : {holidays.shape}')
print(f'Ticket Prices shape: {ticket_prices.shape}')

## 3. Ekstraksi Fitur Endogen (D1–D3 Profil Historis & Skala $S_p$)

Di sini kita mengekstrak karakteristik performa awal:
- Untuk `train`: dari hari penayangan ke 0, 1, 2 di tabel `train`.
- Untuk `test`: dari tabel **`test_history.csv`** (yang mencakup D1-D3 film-film uji).

In [ ]:
def extract_d1_d3_profile(df_hist):
    df_hist = df_hist.sort_values(['movie_title', 'cinema_ids', 'date_show']).copy()
    first_dates = df_hist.groupby(['movie_title', 'cinema_ids'])['date_show'].transform('min')
    df_hist['day_since_release'] = (df_hist['date_show'] - first_dates).dt.days
    d1_d3_subset = df_hist[df_hist['day_since_release'].isin([0, 1, 2])].copy()
    
    # Agregasi Statistik Historis
    prof = d1_d3_subset.groupby(['movie_title', 'cinema_ids']).agg(
        d1_d3_mean=('total_ticket', 'mean'),
        d1_d3_max=('total_ticket', 'max'),
        d1_d3_min=('total_ticket', 'min'),
        d1_d3_sum=('total_ticket', 'sum'),
        d1_d3_std=('total_ticket', 'std'),
        d1_d3_show_mean=('total_show', 'mean'),
        d1_d3_show_max=('total_show', 'max'),
        d1_d3_show_sum=('total_show', 'sum'),
        d1_d3_occ_mean=('occupation_rate', 'mean'),
        d1_d3_occ_max=('occupation_rate', 'max'),
        d1_d3_occ_std=('occupation_rate', 'std')
    ).reset_index()
    
    # Skala MASE Sp = clip(mean(D1..D3), lower=1.0)
    prof['Sp'] = prof['d1_d3_mean'].clip(lower=1.0)
    prof['d1_d3_std'] = prof['d1_d3_std'].fillna(0)
    prof['d1_d3_occ_std'] = prof['d1_d3_occ_std'].fillna(0)
    
    # Pivot Tiket Harian D1, D2, D3
    piv_ticket = d1_d3_subset.pivot_table(index=['movie_title', 'cinema_ids'], columns='day_since_release', values='total_ticket', fill_value=0).reset_index()
    piv_ticket.columns = ['movie_title', 'cinema_ids'] + [f'ticket_d{int(c)+1}' for c in piv_ticket.columns[2:]]
    prof = prof.merge(piv_ticket, on=['movie_title', 'cinema_ids'], how='left')
    
    for col in ['ticket_d1', 'ticket_d2', 'ticket_d3']:
        if col not in prof.columns:
            prof[col] = prof['d1_d3_mean']
        else:
            prof[col] = prof[col].fillna(prof['d1_d3_mean'])
            
    # Momentum & Rasio Pertumbuhan
    prof['d1_d3_trend'] = prof['ticket_d3'] - prof['ticket_d1']
    prof['ratio_d2_d1'] = (prof['ticket_d2'] + 1) / (prof['ticket_d1'] + 1)
    prof['ratio_d3_d2'] = (prof['ticket_d3'] + 1) / (prof['ticket_d2'] + 1)
    prof['ratio_d3_d1'] = (prof['ticket_d3'] + 1) / (prof['ticket_d1'] + 1)
    prof['d1_d3_ticket_per_show'] = (prof['d1_d3_sum'] + 1) / (prof['d1_d3_show_sum'] + 1)
    
    # Tanggal rilis pertama
    rel_dates = d1_d3_subset.groupby(['movie_title', 'cinema_ids'])['date_show'].min().reset_index(name='release_date')
    prof = prof.merge(rel_dates, on=['movie_title', 'cinema_ids'], how='left')
    return prof

train_d1_d3 = extract_d1_d3_profile(train)
test_d1_d3 = extract_d1_d3_profile(test_history)

print(f'Profil D1-D3 Train berhasil dibentuk: {train_d1_d3.shape}')
print(f'Profil D1-D3 Test berhasil dibentuk : {test_d1_d3.shape}')

## 4. Integrasi Fitur Eksogen & Rekayasa Fitur (*Feature Fusion*)

In [ ]:
def build_features(df_target, prof_d1_d3, is_train=True):
    df = df_target.copy()
    
    # Injeksi Profil D1-D3
    df = df.merge(prof_d1_d3, on=['movie_title', 'cinema_ids'], how='left')
    
    # Hitung Hari Sejak Rilis (D4 = 3, D5 = 4, ..., D10 = 9)
    df['day_since_release'] = (df['date_show'] - df['release_date']).dt.days
    
    # Isolasi Data Latih hanya pada cakupan target kompetisi (D4-D10)
    if is_train:
        df = df[df['day_since_release'].isin([3, 4, 5, 6, 7, 8, 9])].copy()
        
    # A. Fitur Kalender
    df['day_of_week'] = df['date_show'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([4, 5, 6]).astype(int)  # Jumat, Sabtu, Minggu
    df['is_sat_sun'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_friday'] = (df['day_of_week'] == 4).astype(int)
    df['day_of_month'] = df['date_show'].dt.day
    df['month'] = df['date_show'].dt.month
    df['is_month_end'] = df['date_show'].dt.is_month_end.astype(int)
    
    # B. Fitur Hari Libur (holidays.csv)
    df = df.merge(holidays[['date', 'day_tipe', 'holiday_tipe']], left_on='date_show', right_on='date', how='left')
    df['is_holiday'] = (df['holiday_tipe'] == 'holiday').astype(int)
    df['is_holiday_weekend'] = df['is_weekend'] * df['is_holiday']
    
    # C. Fitur Harga Tiket (ticket_prices.csv)
    df['extracted_day_tipe'] = np.where(df['day_of_week'] < 4, 'Weekday', np.where(df['day_of_week'] == 4, 'Friday', 'Weekend'))
    df = df.merge(ticket_prices, left_on=['city_name', 'extracted_day_tipe'], right_on=['city_name', 'price_day'], how='left')
    df['price_ceil'] = df['ceil'].fillna(35000)
    df['potential_revenue_est'] = df['price_ceil'] * df['d1_d3_show_mean']
    
    # D. Fitur Metadata Film (movies.csv)
    df = df.merge(movies[['original_title', 'age_rating', 'genre', 'director']], left_on='movie_title', right_on='original_title', how='left')
    df['age_rating'] = df['age_rating'].fillna('Unknown')
    df['genre'] = df['genre'].fillna('Unknown')
    df['director'] = df['director'].fillna('Unknown')
    
    # E. Dinamika Peluruhan & Fitur Interaksi Teoretis
    df['expected_decay'] = 1.0 / (1.0 + 0.15 * (df['day_since_release'] - 2))
    df['decay_sp_interaction'] = df['Sp'] * df['expected_decay']
    df['weekend_sp_interaction'] = df['Sp'] * (1.0 + 0.5 * df['is_weekend'])
    df['day_num'] = df['day_since_release'] + 1
    
    # Bersihkan Kolom Redundan
    drop_cols = ['date_show', 'date', 'price_day', 'extracted_day_tipe', 'original_title', 'release_date', 'day_tipe', 'holiday_tipe', 'ceil']
    df.drop(columns=[c for c in drop_cols if c in df.columns], inplace=True, errors='ignore')
    
    return df

df_train = build_features(train, train_d1_d3, is_train=True)
df_test = build_features(test, test_d1_d3, is_train=False)

# Target Scaling Sesuai Formulasi MASE
df_train['y_scaled'] = df_train['total_ticket'] / df_train['Sp']

# Konversi Kategori
cat_cols = ['cinema_ids', 'city_name', 'movie_title', 'age_rating', 'genre', 'director']
for c in cat_cols:
    df_train[c] = df_train[c].astype('category')
    df_test[c] = df_test[c].astype('category')

feature_cols = [c for c in df_train.columns if c not in ['total_ticket', 'occupation_rate', 'total_show', 'y_scaled']]

print(f'Matriks Train Siap : {df_train.shape}')
print(f'Matriks Test Siap  : {df_test.shape}')
print(f'Jumlah Fitur       : {len(feature_cols)}')
print('Daftar Fitur:', feature_cols)

## 5. Validasi Silang Bebas Leakage (GroupKFold) & Pelatihan Model Ensembel

In [ ]:
gkf = GroupKFold(n_splits=5)

oof_lgb = np.zeros(len(df_train))
oof_cat = np.zeros(len(df_train))
test_preds_lgb = np.zeros(len(df_test))
test_preds_cat = np.zeros(len(df_test))

# Parameter LightGBM (Objective L1)
lgb_params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'n_estimators': 1200,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'random_state': SEED,
    'n_jobs': -1,
    'verbose': -1
}

# Persiapan data string untuk CatBoost
cb_train = df_train[feature_cols].copy()
cb_test = df_test[feature_cols].copy()
for c in cat_cols:
    cb_train[c] = cb_train[c].astype(str)
    cb_test[c] = cb_test[c].astype(str)

print('>> Memulai Pelatihan 5-Fold GroupKFold Ensembel (LightGBM + CatBoost)...')
fold_scores_lgb = []
fold_scores_cat = []
fold_scores_blend = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(df_train, groups=df_train['movie_title'])):
    X_tr, y_tr = df_train.iloc[tr_idx][feature_cols], df_train.iloc[tr_idx]['y_scaled']
    X_va, y_va = df_train.iloc[va_idx][feature_cols], df_train.iloc[va_idx]['y_scaled']
    sp_va = df_train.iloc[va_idx]['Sp'].values
    y_va_raw = df_train.iloc[va_idx]['total_ticket'].values
    
    # 1. Fit LightGBM
    model_lgb = lgb.LGBMRegressor(**lgb_params)
    model_lgb.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], callbacks=[lgb.early_stopping(50, verbose=False)])
    pred_lgb = np.clip(model_lgb.predict(X_va), 0, None)
    oof_lgb[va_idx] = pred_lgb
    test_preds_lgb += np.clip(model_lgb.predict(df_test[feature_cols]), 0, None) / 5
    
    # 2. Fit CatBoost
    cb_tr_x, cb_va_x = cb_train.iloc[tr_idx], cb_train.iloc[va_idx]
    model_cat = CatBoostRegressor(
        loss_function='MAE',
        eval_metric='MAE',
        iterations=1000,
        learning_rate=0.08,
        depth=6,
        cat_features=cat_cols,
        random_seed=SEED,
        verbose=0
    )
    model_cat.fit(cb_tr_x, y_tr, eval_set=(cb_va_x, y_va), early_stopping_rounds=50, verbose=0)
    pred_cat = np.clip(model_cat.predict(cb_va_x), 0, None)
    oof_cat[va_idx] = pred_cat
    test_preds_cat += np.clip(model_cat.predict(cb_test), 0, None) / 5
    
    # 3. Evaluasi MASE per Fold
    mase_lgb = np.mean(np.abs(y_va_raw - pred_lgb * sp_va) / sp_va)
    mase_cat = np.mean(np.abs(y_va_raw - pred_cat * sp_va) / sp_va)
    pred_blend = 0.6 * pred_lgb + 0.4 * pred_cat
    mase_blend = np.mean(np.abs(y_va_raw - pred_blend * sp_va) / sp_va)
    
    fold_scores_lgb.append(mase_lgb)
    fold_scores_cat.append(mase_cat)
    fold_scores_blend.append(mase_blend)
    
    print(f'Fold {fold+1} | MASE LGB: {mase_lgb:.4f} | MASE Cat: {mase_cat:.4f} | MASE Blend: {mase_blend:.4f}')

sp_all = df_train['Sp'].values
y_raw_all = df_train['total_ticket'].values

total_mase_lgb = np.mean(np.abs(y_raw_all - oof_lgb * sp_all) / sp_all)
total_mase_cat = np.mean(np.abs(y_raw_all - oof_cat * sp_all) / sp_all)
oof_blend = 0.6 * oof_lgb + 0.4 * oof_cat
total_mase_blend = np.mean(np.abs(y_raw_all - oof_blend * sp_all) / sp_all)

print('='*60)
print(f'✅ SKOR FINAL 5-FOLD CV MASE (LightGBM) : {total_mase_lgb:.4f}')
print(f'✅ SKOR FINAL 5-FOLD CV MASE (CatBoost) : {total_mase_cat:.4f}')
print(f'🏆 SKOR FINAL 5-FOLD CV MASE (ENSEMBLE) : {total_mase_blend:.4f}')
print('='*60)

## 6. Evaluasi & Feature Importance Analysis

In [ ]:
plt.figure(figsize=(12, 8))
feat_imp = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model_lgb.feature_importances_
}).sort_values('Importance', ascending=False).head(20)

sns.barplot(data=feat_imp, x='Importance', y='Feature', palette='viridis')
plt.title('Top 20 Fitur Paling Berpengaruh (LightGBM MASE Formulation)', fontsize=14, fontweight='bold')
plt.xlabel('Tingkat Kepentingan Fitur (Split Count)')
plt.tight_layout()
plt.show()

## 7. Inference, Post-Processing, & Submission File Assembly

In [ ]:
# 1. Ensembling Prediksi Uji pada Skala Terscaling
final_test_scaled = 0.6 * test_preds_lgb + 0.4 * test_preds_cat

# 2. Pengembalian Skala Target: y_pred = y_scaled * Sp
test_sp = df_test['Sp'].values
final_test_pred = final_test_scaled * test_sp

# 3. Post-Processing: Batas bawah logis (non-negatif) & pembulatan tiket bulat
final_test_pred = np.clip(final_test_pred, 0, None)
final_test_pred = np.round(final_test_pred).astype(int)

# 4. Pembentukan Submission File
sub_template_path = os.path.join(data_dir, 'sample_submission.csv')
sub = pd.read_csv(sub_template_path)

if len(sub) != len(final_test_pred):
    raise ValueError(f'Ukuran submission ({len(sub)}) tidak cocok dengan hasil prediksi ({len(final_test_pred)})!')

sub['total_ticket'] = final_test_pred

output_dir = 'submission'
os.makedirs(output_dir, exist_ok=True)
output_file = os.path.join(output_dir, 'Exp02_Submission_MASE.csv')
sub.to_csv(output_file, index=False)

print('='*65)
print(f'✅ [SUKSES BESAR] File Submission Resmi Tercetak di: {output_file}')
print(f'Total Baris Prediksi : {len(sub)}')
print('Ringkasan Statistik Prediksi Tiket:')
print(sub['total_ticket'].describe())
print('='*65)
print(sub.head(10))